### Tahap 1: Data Cleaning, Merging, dan Segmentasi Usia Pengunjung

Notebook ini memuat ketiga dataset wisata dari repositori GitHub (`project-SainsData`), membersihkan nilai kosong/duplikat, serta menggabungkan data rating dan profil pengguna untuk menghasilkan *insight* demografi pengunjung (**Remaja**, **Muda**, dan **Tua**) pada setiap destinasi wisata.

In [8]:
import numpy as np
import pandas as pd

# 1. Load Ketiga Dataset Langsung dari GitHub (Online)
base_url = "https://raw.githubusercontent.com/n00rah578/project-SainsData/main/dataSETFIKS"

df_places = pd.read_csv(f"{base_url}/tourism_with_id.csv")
df_rating = pd.read_csv(f"{base_url}/tourism_rating.csv")
df_user = pd.read_csv(f"{base_url}/user.csv")

print("=== 1. DATA BERHASIL DIMUAT DARI GITHUB ===")
print(f"Places: {df_places.shape} | Rating: {df_rating.shape} | User: {df_user.shape}")

=== 1. DATA BERHASIL DIMUAT DARI GITHUB ===
Places: (437, 13) | Rating: (10000, 3) | User: (300, 3)


### 2. Data Cleaning
Membersihkan kolom kosong bawaan (`Unnamed: 11`, `Unnamed: 12`), mengisi *missing values* pada `Time_Minutes` menggunakan nilai median, serta menghapus baris duplikat pada data rating.

In [9]:
# Hapus kolom kosong bawaan dataset
cols_to_drop = [col for col in ['Unnamed: 11', 'Unnamed: 12'] if col in df_places.columns]
df_places_clean = df_places.drop(columns=cols_to_drop).copy()

# Isi missing values Time_Minutes dengan median
if 'Time_Minutes' in df_places_clean.columns:
    df_places_clean['Time_Minutes'] = df_places_clean['Time_Minutes'].fillna(
        df_places_clean['Time_Minutes'].median()
    )

# Hapus duplikat pada rating jika ada
df_rating_clean = df_rating.drop_duplicates().copy()

print("Data cleaning selesai!")
print(f"Jumlah data tempat bersih : {df_places_clean.shape}")
print(f"Jumlah data rating bersih : {df_rating_clean.shape}")

Data cleaning selesai!
Jumlah data tempat bersih : (437, 11)
Jumlah data rating bersih : (9921, 3)


### 3. Penggabungan Data (Merging) & Segmentasi Kategori Usia
Menggabungkan tabel interaksi rating dengan usia pengguna (`user.csv`) untuk mengelompokkan pengunjung tiap tempat wisata ke dalam 3 kategori usia (berdasarkan rentang aktual 18–40 tahun):
- **Remaja (18–22 tahun)**
- **Muda (23–30 tahun)**
- **Tua (31–40 tahun)**

In [10]:
# Gabungkan rating dengan data user dan tempat wisata
df_merged = (
    df_rating_clean
    .merge(df_user, on='User_Id', how='left')
    .merge(df_places_clean[['Place_Id', 'Place_Name', 'Category', 'City']], on='Place_Id', how='left')
)

# Fungsi kategorisasi usia (sesuai rentang dataset 18-40 tahun)
def kategori_usia(umur):
    if umur <= 22:
        return 'Remaja (18-22)'
    elif umur <= 30:
        return 'Muda (23-30)'
    else:
        return 'Tua (31-40)'

df_merged['Kategori_Usia'] = df_merged['Age'].apply(kategori_usia)

# Hitung jumlah pengunjung tiap kategori usia per tempat wisata
demografi_tempat = (
    df_merged.groupby(['Place_Id', 'Kategori_Usia'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

kolom_usia = ['Remaja (18-22)', 'Muda (23-30)', 'Tua (31-40)']
for col in kolom_usia:
    if col not in demografi_tempat.columns:
        demografi_tempat[col] = 0

demografi_tempat['Total_Pengunjung'] = demografi_tempat[kolom_usia].sum(axis=1)
demografi_tempat['Dominasi_Pengunjung'] = demografi_tempat[kolom_usia].idxmax(axis=1)

# Gabungkan kembali statistik demografi ke tabel master tempat wisata
df_places_final = df_places_clean.merge(demografi_tempat, on='Place_Id', how='left')
df_places_final[kolom_usia + ['Total_Pengunjung']] = (
    df_places_final[kolom_usia + ['Total_Pengunjung']].fillna(0).astype(int)
)
df_places_final['Dominasi_Pengunjung'] = df_places_final['Dominasi_Pengunjung'].fillna('Belum Ada Data')

# Preview hasil penggabungan & segmentasi
df_places_final[['Place_Id', 'Place_Name', 'Remaja (18-22)', 'Muda (23-30)', 'Tua (31-40)', 'Total_Pengunjung', 'Dominasi_Pengunjung']].head(10)

,Place_Id,Place_Name,Remaja (18-22),Muda (23-30),Tua (31-40),Total_Pengunjung,Dominasi_Pengunjung
0,1,Monumen Nasional,6,7,5,18,Muda (23-30)
1,2,Kota Tua,5,9,11,25,Tua (31-40)
2,3,Dunia Fantasi,5,7,7,19,Muda (23-30)
3,4,Taman Mini Indonesia Indah (TMII),4,7,10,21,Tua (31-40)
4,5,Atlantis Water Adventure,4,14,7,25,Muda (23-30)
5,6,Taman Impian Jaya Ancol,4,8,12,24,Tua (31-40)
6,7,Kebun Binatang Ragunan,5,7,9,21,Tua (31-40)
7,8,Ocean Ecopark,6,11,13,30,Tua (31-40)
8,9,Pelabuhan Marina,5,6,8,19,Tua (31-40)
9,10,Pulau Tidung,1,4,9,14,Tua (31-40)


### 4. Ekspor Dataset Bersih
Menyimpan hasil akhir menjadi `places_with_demographics.csv` dan `rating_clean.csv` untuk digunakan oleh **ML Engineer** (pelatihan model SVD) dan **Web Developer** (tampilan Streamlit).

In [11]:
df_places_final.to_csv('places_with_demographics.csv', index=False)
df_rating_clean.to_csv('rating_clean.csv', index=False)

print("Selesai! File 'places_with_demographics.csv' dan 'rating_clean.csv' berhasil disimpan.")

Selesai! File 'places_with_demographics.csv' dan 'rating_clean.csv' berhasil disimpan.
